# PBISC floor read-out: disease DEG recomputation and rare-population expression (2026-10-27)

Inputs (all local, nothing regenerated):
- `ilisi_floor_20261021/data/pb/*.npz` — per-sample x six-type pseudobulk (raw count sums, full files), built by `pb_build.py`
- `ilisi_floor_20261021/data/big/*.h5ad` — 60k-cell balanced subsets (real half A = `ref`, real half B = `realU`, `floor`, `prev4k`)
- `ilisi_floor_20261021/results/big/*_transfer.csv.gz` — kNN-transferred labels for real B and floor

Contrasts: sepsis = adult sepsis vs adult healthy control; COVID = positive vs negative, aggregated to patient level.
DEG rule: Welch t-test on log2 CPM pseudobulk, BH FDR < 0.05 and |log2FC| >= 1. Groups with < 20 cells dropped.
COVID floor excludes the 5 adapter-training-donor samples.

## Library: pseudobulk loading, design, DEG

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
PB="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/pb/"
COVID_EXCL={"C126_positive_5","C126_positive_9","C126_positive_11","C141_negative","C58_positive_2"}
SIX=["T cell","B cell","NK cell","Monocyte","DC"]
MINCELL=20

def bh(p):
    p=np.asarray(p,float); ok=~np.isnan(p); q=np.full(p.shape,np.nan)
    pv=p[ok]; n=len(pv); o=np.argsort(pv); r=np.arange(1,n+1)
    qv=np.minimum.accumulate((pv[o]*n/r)[::-1])[::-1]
    out=np.empty(n); out[o]=np.minimum(qv,1); q[ok]=out; return q

def load(name):
    z=np.load(PB+name+".npz",allow_pickle=True)
    S=z["S"].astype(np.float64); n=z["n"]; groups=z["groups"].astype(str); genes=z["genes"].astype(str)
    samp=np.array([g.split("||")[0] for g in groups]); six=np.array([g.split("||")[1] for g in groups])
    obs=pd.read_csv(PB+name+"_obs.csv")
    return dict(S=S,n=n,samp=samp,six=six,genes=genes,obs=obs)

def design(name,d):
    """return sample -> (unit, condition) ; condition in {'case','ctrl'} else None"""
    o=d["obs"].copy()
    if name.endswith("sepsis"):
        if "age_group" in o:  # real
            o=o[o.age_group=="adult"]
            m={"sepsis":"case","healthy control":"ctrl"}
            o["cond"]=o.group.map(m)
        else:
            m={"adult_sepsis":"case","adult_HC":"ctrl"}
            o["cond"]=o.group.map(m)
        o["unit"]=o["sample"]
    else:
        if "case_control" in o:  # real covid
            m={"POSITIVE":"case","NEGATIVE":"ctrl"}
            o["cond"]=o.case_control.map(m); o["unit"]=o["patient"]
        else:
            o=o[~o["sample"].isin(COVID_EXCL)]
            m={"covid_positive":"case","covid_negative":"ctrl"}
            o["cond"]=o.group.map(m)
            o["unit"]=o["sample"].str.split("_").str[0]
    o=o.dropna(subset=["cond"])
    return o[["sample","unit","cond"]]

def cpm_matrix(d,des,ct):
    """samples x genes log2CPM for one six_type, then averaged to unit level"""
    sel=(d["six"]==ct)
    S=d["S"][sel]; n=d["n"][sel]; samp=d["samp"][sel]
    keep=n>=MINCELL
    S,samp=S[keep],samp[keep]
    tot=S.sum(1,keepdims=True); tot[tot==0]=1
    L=np.log2(S/tot*1e6+1)
    df=pd.DataFrame(L,index=samp,columns=d["genes"])
    df=df.loc[df.index.isin(des["sample"])]
    mp=des.set_index("sample")
    df["_u"]=mp.loc[df.index,"unit"].values; df["_c"]=mp.loc[df.index,"cond"].values
    g=df.groupby(["_u","_c"]).mean()
    g=g.reset_index()
    return g

def deg(g,genes,lfc=1.0,q=0.05):
    a=g[g._c=="case"][genes].to_numpy(); b=g[g._c=="ctrl"][genes].to_numpy()
    if len(a)<3 or len(b)<3: return None
    lf=a.mean(0)-b.mean(0)
    t,p=stats.ttest_ind(a,b,equal_var=False)
    qv=bh(p)
    return pd.DataFrame(dict(gene=genes,logFC=lf,p=p,q=qv,
                             mean_case=a.mean(0),mean_ctrl=b.mean(0),
                             n_case=len(a),n_ctrl=len(b),
                             deg=(qv<q)&(np.abs(lf)>=lfc)))


## 1. Disease DEG, real vs floor vs prev K4000

In [ ]:
import numpy as np, pandas as pd, json, os
from scipy import stats
from deg_lib import *
os.makedirs("out",exist_ok=True)
res={}; allde={}
for coh in ["sepsis","covid"]:
    srcs={s:load(f"{s}_{coh}") for s in ["real","floor","prev"]}
    genes=srcs["real"]["genes"]
    # measured genes: non-zero in real somewhere AND non-zero in floor somewhere
    meas=(srcs["real"]["S"].sum(0)>0)&(srcs["floor"]["S"].sum(0)>0)
    gm=genes[meas]
    des={s:design(f"{s}_{coh}",srcs[s]) for s in srcs}
    for ct in SIX:
        tabs={}
        for s in srcs:
            g=cpm_matrix(srcs[s],des[s],ct)
            if len(g)==0: continue
            t=deg(g,list(genes))
            if t is None: continue
            t=t[t.gene.isin(gm)].reset_index(drop=True)
            tabs[s]=t
        if "real" not in tabs: continue
        allde[(coh,ct)]=tabs
        R=tabs["real"]
        # expression rank in real (case+ctrl mean)
        rexp=(R.mean_case+R.mean_ctrl)/2
        rank=pd.Series(stats.rankdata(-rexp),index=R.gene)
        for s in ["floor","prev"]:
            if s not in tabs: continue
            F=tabs[s].set_index("gene").loc[R.gene]
            row=dict(cohort=coh,celltype=ct,source=s,
                     n_case_real=int(R.n_case[0]),n_ctrl_real=int(R.n_ctrl[0]),
                     n_case_gen=int(F.n_case.iloc[0]),n_ctrl_gen=int(F.n_ctrl.iloc[0]),
                     n_genes=len(R))
            row["logFC_r"]=float(np.corrcoef(R.logFC,F.logFC)[0,1])
            row["logFC_rho"]=float(stats.spearmanr(R.logFC,F.logFC).statistic)
            sl=np.polyfit(R.logFC.values,F.logFC.values,1)[0]; row["slope"]=float(sl)
            hi=(rank.values>2000)
            if hi.sum()>50:
                row["logFC_r_rank2000plus"]=float(np.corrcoef(R.logFC[hi],F.logFC[hi])[0,1])
                row["slope_rank2000plus"]=float(np.polyfit(R.logFC[hi],F.logFC[hi],1)[0])
            rd=R.deg.values; fd=F.deg.values
            row["n_DEG_real"]=int(rd.sum()); row["n_DEG_gen"]=int(fd.sum())
            row["recall"]=float((rd&fd).sum()/max(rd.sum(),1)) if rd.sum() else np.nan
            row["precision"]=float((rd&fd).sum()/max(fd.sum(),1)) if fd.sum() else np.nan
            row["jaccard"]=float((rd&fd).sum()/max((rd|fd).sum(),1))
            row["F1"]=float(2*row["recall"]*row["precision"]/(row["recall"]+row["precision"])) if (rd.sum() and fd.sum() and (row["recall"]+row["precision"])>0) else np.nan
            if rd.sum():
                row["sign_agree_realDEG"]=float((np.sign(R.logFC[rd])==np.sign(F.logFC.values[rd])).mean())
            row["sign_agree_all"]=float((np.sign(R.logFC)==np.sign(F.logFC.values)).mean())
            # top-100 by |logFC| among q<0.05
            def top100(T):
                t=T[T.q<0.05]
                return set(t.reindex(t.logFC.abs().sort_values(ascending=False).index).gene.head(100))
            tr=top100(R); tf=top100(F.reset_index())
            row["top100_overlap"]=len(tr&tf)
            res[(coh,ct,s)]=row
D=pd.DataFrame(res.values())
D.to_csv("out/deg_summary.csv",index=False)
pd.set_option("display.width",250)
print(D[["cohort","celltype","source","n_case_real","n_ctrl_real","n_case_gen","n_ctrl_gen","n_DEG_real","n_DEG_gen","logFC_r","logFC_r_rank2000plus","slope","sign_agree_realDEG","recall","precision","top100_overlap"]].round(3).to_string(index=False))
import pickle
pickle.dump(allde,open("out/allde.pkl","wb"))


## 2. Real-vs-real ceiling (split-half of the real samples, 20 reps)

In [ ]:
import numpy as np, pandas as pd, pickle
from scipy import stats
from deg_lib import *
rng=np.random.default_rng(20261027)
rows=[]
for coh in ["sepsis","covid"]:
    d=load(f"real_{coh}"); des=design(f"real_{coh}",d); genes=d["genes"]
    fl=load(f"floor_{coh}")
    meas=(d["S"].sum(0)>0)&(fl["S"].sum(0)>0); gm=set(genes[meas])
    for ct in SIX:
        g=cpm_matrix(d,des,ct)
        if len(g)==0: continue
        case=g[g._c=="case"]._u.unique(); ctrl=g[g._c=="ctrl"]._u.unique()
        if len(case)<8 or len(ctrl)<6: 
            rows.append(dict(cohort=coh,celltype=ct,note=f"too few units case={len(case)} ctrl={len(ctrl)}")); continue
        acc=[]
        for rep in range(20):
            ca=rng.permutation(case); co=rng.permutation(ctrl)
            h=[(ca[:len(ca)//2],co[:len(co)//2]),(ca[len(ca)//2:],co[len(co)//2:])]
            ts=[]
            for cs,cl in h:
                sub=g[g._u.isin(list(cs)+list(cl))]
                t=deg(sub,list(genes))
                if t is None: ts=[]; break
                ts.append(t[t.gene.isin(gm)].reset_index(drop=True))
            if len(ts)!=2: continue
            A,B=ts
            r=np.corrcoef(A.logFC,B.logFC)[0,1]
            rec=(A.deg&B.deg).sum()/max(A.deg.sum(),1) if A.deg.sum() else np.nan
            pre=(A.deg&B.deg).sum()/max(B.deg.sum(),1) if B.deg.sum() else np.nan
            sa=(np.sign(A.logFC[A.deg])==np.sign(B.logFC[A.deg])).mean() if A.deg.sum() else np.nan
            acc.append((r,rec,pre,sa,A.deg.sum(),B.deg.sum()))
        if not acc: continue
        a=np.array(acc,float)
        rows.append(dict(cohort=coh,celltype=ct,n_case=len(case),n_ctrl=len(ctrl),reps=len(acc),
                         logFC_r=np.nanmean(a[:,0]),logFC_r_sd=np.nanstd(a[:,0]),
                         recall=np.nanmean(a[:,1]),precision=np.nanmean(a[:,2]),
                         sign_agree=np.nanmean(a[:,3]),nDEG_halfA=np.nanmean(a[:,4]),nDEG_halfB=np.nanmean(a[:,5])))
C=pd.DataFrame(rows); C.to_csv("out/real_real_ceiling.csv",index=False)
pd.set_option("display.width",250)
print(C.round(3).to_string(index=False))


## 3. Rank-matched metrics and top DEG lists

In [ ]:
import numpy as np, pandas as pd, pickle
from scipy import stats
from deg_lib import *
allde=pickle.load(open("out/allde.pkl","rb"))
rows=[]; tops=[]
for (coh,ct),tabs in allde.items():
    R=tabs["real"]; rexp=(R.mean_case+R.mean_ctrl)/2
    rank=stats.rankdata(-rexp); hi=rank>2000
    realDEG=set(R.gene[R.deg])
    for s in ["floor","prev"]:
        if s not in tabs: continue
        F=tabs[s].set_index("gene").loc[R.gene].reset_index()
        r=dict(cohort=coh,celltype=ct,source=s)
        r["sign_rank2000plus"]=float((np.sign(R.logFC[hi])==np.sign(F.logFC[hi])).mean())
        r["slope_rank2000plus"]=float(np.polyfit(R.logFC[hi],F.logFC[hi],1)[0])
        # precision of generated top-100 DEGs (|logFC| ranked among q<0.05) vs real DEG set
        t=F[F.q<0.05]
        top=t.reindex(t.logFC.abs().sort_values(ascending=False).index).gene.head(100).tolist()
        r["gen_top100_n"]=len(top)
        r["gen_top100_precision"]=float(np.mean([g in realDEG for g in top])) if top else np.nan
        # same-direction among those
        lr=R.set_index("gene").logFC; lf=F.set_index("gene").logFC
        r["gen_top100_signmatch"]=float(np.mean([np.sign(lf[g])==np.sign(lr[g]) for g in top])) if top else np.nan
        rows.append(r)
    # top DEG lists
    def tl(T,k=30):
        t=T[T.q<0.05]
        t=t.reindex(t.logFC.abs().sort_values(ascending=False).index).head(k)
        return list(zip(t.gene,t.logFC.round(2)))
    tops.append(dict(cohort=coh,celltype=ct,
                     real=tl(R), floor=tl(tabs["floor"]) if "floor" in tabs else [],
                     prev=tl(tabs["prev"]) if "prev" in tabs else []))
S=pd.DataFrame(rows); S.to_csv("out/deg_rank2000_summary.csv",index=False)
pd.set_option("display.width",250)
print(S.round(3).to_string(index=False))
print()
print("=== aggregate (median over 5 six-types; and excluding DC) ===")
D=pd.read_csv("out/deg_summary.csv")
for coh in ["sepsis","covid"]:
    for s in ["prev","floor"]:
        d=D[(D.cohort==coh)&(D.source==s)]; dn=d[d.celltype!="DC"]
        q=S[(S.cohort==coh)&(S.source==s)]; qn=q[q.celltype!="DC"]
        print(f"{coh:7s} {s:5s} logFC_r med {d.logFC_r.median():.2f} / exDC mean {dn.logFC_r.mean():.2f} | "
              f"rank2000+ r exDC {dn.logFC_r_rank2000plus.mean():.2f} | slope exDC {qn.slope_rank2000plus.mean():.2f} | "
              f"sign rank2000+ exDC {qn.sign_rank2000plus.mean():.2f} | gen top100 prec exDC {qn.gen_top100_precision.mean():.2f}")
pickle.dump(tops,open("out/tops.pkl","wb"))
for t in tops:
    print(f"\n--- {t['cohort']} / {t['celltype']}")
    print("  real :", ", ".join(f"{g}({v:+.1f})" for g,v in t["real"][:15]))
    print("  floor:", ", ".join(f"{g}({v:+.1f})" for g,v in t["floor"][:15]))


## 4. Named genes from the manuscript / source papers

In [ ]:
import numpy as np, pandas as pd, pickle
allde=pickle.load(open("out/allde.pkl","rb"))
PANEL={
 ("sepsis","Monocyte"):["IL1R2","MCEMP1","AQP9","TNFAIP6","ACSL1","S100A8","S100A9","HLA-DRA","HLA-DRB1","HLA-DPA1","CD74","IFI27","C1QA","C1QB","C1QC","VSIG4","CLU","RNASE1","PLD4","CXCL1","CCL20","ADAMTS2","RETN","SERPINB2"],
 ("covid","Monocyte"):["IFI27","ISG15","IFI6","IFI44L","IFIT1","IFIT3","MX1","OAS1","SIGLEC1","LY6E","HLA-DRA","CD74","S100A8","S100A9"],
 ("covid","T cell"):["IFI27","ISG15","IFI6","IFI44L","IFIT1","IFIT3","MX1","OAS1","LY6E","S100A8","S100A9"],
 ("covid","NK cell"):["IFI27","ISG15","IFI6","IFI44L","IFIT1","IFIT3","MX1","OAS1","LY6E","S100A8","S100A9"],
 ("covid","B cell"):["IFI27","ISG15","IFI6","IFI44L","IFIT1","IFIT3","MX1","OAS1","LY6E","S100A8","S100A9"],
 ("covid","DC"):["IFI27","ISG15","IFI6","IFI44L","IFIT1","IFIT3","MX1","OAS1","LY6E","S100A8","S100A9"],
 ("sepsis","T cell"):["NR4A1","NR4A2","NR4A3","ATF3","EGR1","SOCS3","DUSP1","CDKN1A","HBB","SLC4A10","ZNF683","S100A8","S100A9"],
 ("sepsis","NK cell"):["NR4A1","ATF3","SOCS3","EGR1","HBB","GNLY","NKG7","PRF1","S100A8","S100A9"],
 ("sepsis","B cell"):["NR4A1","SOCS3","TNF","MZB1","JCHAIN","S100A8","S100A9"],
 ("sepsis","DC"):["FCER1A","LILRA4","GZMB","PTGDS","CLIC3","CD1C","IRF7","IL1R2"],
}
rows=[]
for (coh,ct),genes in PANEL.items():
    if (coh,ct) not in allde: continue
    tabs=allde[(coh,ct)]
    R=tabs["real"].set_index("gene")
    for s in ["floor","prev"]:
        if s not in tabs: continue
        F=tabs[s].set_index("gene")
        for g in genes:
            if g not in R.index or g not in F.index: 
                rows.append(dict(cohort=coh,celltype=ct,source=s,gene=g,note="not in panel")); continue
            rows.append(dict(cohort=coh,celltype=ct,source=s,gene=g,
                real_logFC=round(float(R.logFC[g]),2), real_q=float(R.q[g]), real_DEG=bool(R.deg[g]),
                gen_logFC=round(float(F.logFC[g]),2), gen_q=float(F.q[g]), gen_DEG=bool(F.deg[g]),
                ratio=round(float(F.logFC[g]/R.logFC[g]),2) if abs(R.logFC[g])>0.2 else np.nan,
                sign_ok=bool(np.sign(R.logFC[g])==np.sign(F.logFC[g]))))
N=pd.DataFrame(rows); N.to_csv("out/named_genes.csv",index=False)
pd.set_option("display.width",260); pd.set_option("display.max_rows",400)
f=N[N.source=="floor"]
for coh,ct in [("sepsis","Monocyte"),("sepsis","DC"),("covid","Monocyte"),("covid","T cell")]:
    d=f[(f.cohort==coh)&(f.celltype==ct)]
    print(f"\n### {coh} / {ct}  (real vs floor)")
    print(d[["gene","real_logFC","real_DEG","gen_logFC","gen_DEG","ratio","sign_ok"]].to_string(index=False))
print("\n=== IFN panel magnitude ratio (floor/real), COVID, by cell type ===")
ifn=["IFI27","ISG15","IFI6","IFI44L","IFIT1","IFIT3","MX1","OAS1","LY6E"]
d=f[(f.cohort=="covid")&(f.gene.isin(ifn))]
print(d.pivot_table(index="gene",columns="celltype",values="ratio").round(2).to_string())
print("\n=== S100A8/A9 leakage into lymphoid lineages (COVID) ===")
d=f[(f.cohort=="covid")&(f.gene.isin(["S100A8","S100A9"]))]
print(d[["celltype","gene","real_logFC","real_DEG","gen_logFC","gen_DEG"]].to_string(index=False))


## Library: rare-population definitions (same as f_rare_parent.py / big_rare.py)

In [ ]:
import numpy as np, pandas as pd, anndata as ad, scipy.sparse as sp
IN="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/big/"
TR="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/results/big/"
st=lambda m,*p: np.any([np.char.startswith(np.asarray(m,dtype=str),q) for q in p],axis=0)
SEPSIS={"Mono_c06-CD14-MKI67":lambda m,l:m=="Monocytes","B_c08-MKI67":lambda m,l:np.isin(m,["B cells","Plasma cells"]),
 "T_CD4_c07-MKI67":lambda m,l:m=="CD4+ T cells","T_CD8_c06-MKI67":lambda m,l:m=="CD8+ T cells","NK_c04-MKI67":lambda m,l:m=="NK cells",
 "B_c07-MZB1":lambda m,l:np.isin(m,["B cells","Plasma cells"]),"pDC":lambda m,l:m=="DC","cDC":lambda m,l:m=="DC",
 "Macrophage":lambda m,l:np.isin(m,["Monocytes","Macrophages"]),"gdT":lambda m,l:np.isin(m,["CD4+ T cells","CD8+ T cells","NKT cells"])|(l=="gdT"),
 "ILC":lambda m,l:(m=="NK cells")|(l=="ILC"),"HSC":lambda m,l:np.ones(len(m),bool),
 "Mono_c01-CD14-HLA-DRA":lambda m,l:m=="Monocytes","NKT_c05-IFNG-CCL4hi":lambda m,l:m=="NKT cells"}
PAR_S={"Mono_c06-CD14-MKI67":"Monocytes","B_c08-MKI67":"B + plasma","T_CD4_c07-MKI67":"CD4 T","T_CD8_c06-MKI67":"CD8 T","NK_c04-MKI67":"NK",
 "B_c07-MZB1":"B + plasma","pDC":"DC","cDC":"DC","Macrophage":"Mono + macrophage","gdT":"all T","ILC":"NK + ILC","HSC":"all cells",
 "Mono_c01-CD14-HLA-DRA":"Monocytes","NKT_c05-IFNG-CCL4hi":"NKT"}
COVID={"CD14mono_dex":lambda m,l:st(m,"CD14mono"),"CD14mono_IFN":lambda m,l:st(m,"CD14mono"),"B_ASC_IgG":lambda m,l:st(m,"B_ASC"),
 "B_ASC_dividing":lambda m,l:st(m,"B_ASC"),"T CD4 cycling":lambda m,l:st(m,"T CD4","T reg"),"ASDC":lambda m,l:st(m,"DC","ASDC","pDC"),
 "DC1":lambda m,l:st(m,"DC","ASDC","pDC"),"pDC":lambda m,l:st(m,"DC","ASDC","pDC"),"HSPC":lambda m,l:np.ones(len(m),bool),
 "ILC":lambda m,l:st(m,"NK","ILC"),"T g/d":lambda m,l:st(m,"T ","MAIT","NKT"),"MAIT":lambda m,l:st(m,"T ","MAIT","NKT"),
 "NK CD56 bright":lambda m,l:st(m,"NK")}
PAR_C={"CD14mono_dex":"CD14 mono","CD14mono_IFN":"CD14 mono","B_ASC_IgG":"plasmablasts","B_ASC_dividing":"plasmablasts","T CD4 cycling":"CD4 T",
 "ASDC":"DC","DC1":"DC","pDC":"DC","HSPC":"all cells","ILC":"NK + ILC","T g/d":"all T","MAIT":"all T","NK CD56 bright":"NK"}
def lognorm(X):
    X=sp.csr_matrix(X,copy=True).astype(np.float64)
    t=np.asarray(X.sum(1)).ravel(); t[t==0]=1
    X=sp.diags(1e4/t)@X; X.data=np.log1p(X.data); return sp.csr_matrix(X)
def load_coh(coh):
    R=ad.read_h5ad(f"{IN}{coh}_ref.h5ad"); U=ad.read_h5ad(f"{IN}{coh}_realU.h5ad"); F=ad.read_h5ad(f"{IN}{coh}_floor.h5ad")
    tr={s:pd.read_csv(f"{TR}{coh}_{s}_transfer.csv.gz",index_col=0,keep_default_na=False) for s in ("realU","floor")}
    genes=np.array(R.var_names)
    X={"refA":lognorm(R.X),"realB":lognorm(U.X),"floor":lognorm(F.X)}
    V={"refA":(R.obs.major.astype(str).to_numpy(),R.obs.fine.astype(str).to_numpy()),
       "realB_auth":(tr["realU"].major.to_numpy(),tr["realU"].fine.to_numpy()),
       "realB_tr":(tr["realU"].major_tr.to_numpy(),tr["realU"].fine_tr.to_numpy()),
       "floor_tr":(tr["floor"].major_tr.to_numpy(),tr["floor"].fine_tr.to_numpy())}
    SRC={"refA":"refA","realB_auth":"realB","realB_tr":"realB","floor_tr":"floor"}
    return X,V,SRC,genes,(SEPSIS,PAR_S) if coh=="sepsis" else (COVID,PAR_C)


## 5. Rare-population expression: profile correlation and marker programme

In [ ]:
import numpy as np, pandas as pd, pickle
from scipy import stats
from rare_lib import *
rng=np.random.default_rng(20261027)
MIN=30
rows=[]; genediff=[]
for coh in ["sepsis","covid"]:
    X,V,SRC,genes,(DEF,PAR)=load_coh(coh)
    mA,fA=V["refA"]
    # expressed-gene filter from real A
    detA=np.asarray((X["refA"]>0).mean(0)).ravel()
    gok=detA>=0.01
    for sub,fn in DEF.items():
        parA=fn(mA,fA); tgA=parA&(fA==sub)
        if tgA.sum()<MIN: continue
        A=X["refA"]
        # top-10 markers from real A (subset vs rest of parent)
        d=np.asarray(A[tgA].mean(0)).ravel()-np.asarray(A[parA&~tgA].mean(0)).ravel()
        det=np.asarray((A[tgA]>0).mean(0)).ravel(); d2=d.copy(); d2[det<0.1]=-np.inf
        top=np.argsort(-d2)[:10]; mk=genes[top]
        prof={"refA":np.asarray(A[tgA].mean(0)).ravel()}
        ncell={"refA":int(tgA.sum())}
        for lab in ["realB_tr","floor_tr"]:
            m,f=V[lab]; p=fn(m,f); t=p&(f==sub)
            ncell[lab]=int(t.sum())
            prof[lab]=np.asarray(X[SRC[lab]][t].mean(0)).ravel() if t.sum()>0 else None
        if prof["floor_tr"] is None or ncell["floor_tr"]<MIN: 
            rows.append(dict(cohort=coh,subset=sub,parent=PAR[sub],n_refA=ncell["refA"],n_realB_tr=ncell["realB_tr"],n_floor_tr=ncell["floor_tr"],note="floor n<30")); continue
        r=dict(cohort=coh,subset=sub,parent=PAR[sub],n_refA=ncell["refA"],n_realB_tr=ncell["realB_tr"],n_floor_tr=ncell["floor_tr"],markers=",".join(mk))
        a=prof["refA"][gok]
        for lab,nm in [("realB_tr","realB"),("floor_tr","floor")]:
            if prof[lab] is None or ncell[lab]<10: continue
            b=prof[lab][gok]
            r[f"r_{nm}"]=float(np.corrcoef(a,b)[0,1])
            r[f"rho_{nm}"]=float(stats.spearmanr(a,b).statistic)
            diff=b-a
            r[f"nDiff_{nm}"]=int((np.abs(diff)>=0.5).sum())
            r[f"meanAbsDiff_{nm}"]=float(np.abs(diff).mean())
        # marker score level within subset, and detection
        for lab,nm in [("refA","refA"),("realB_tr","realB"),("floor_tr","floor")]:
            m,f=(V[lab] if lab!="refA" else V["refA"]); p=fn(m,f); t=p&(f==sub) if lab!="refA" else tgA
            if t.sum()<10: continue
            Z=X[SRC.get(lab,"refA")] if lab!="refA" else X["refA"]
            r[f"mkScore_{nm}"]=float(np.asarray(Z[t][:,top].mean(1)).ravel().mean())
            r[f"mkDet_{nm}"]=float(np.asarray((Z[t][:,top]>0).mean(0)).ravel().mean())
            # identity AUROC within parent
            sc=np.asarray(Z[:,top].mean(1)).ravel()
            if 0<t[p].sum()<p.sum():
                from sklearn.metrics import roc_auc_score
                r[f"auroc_{nm}"]=float(roc_auc_score(t[p],sc[p]))
        rows.append(r)
        # per-gene differences for the biggest subsets
        if ncell["floor_tr"]>=100:
            dd=pd.DataFrame(dict(gene=genes[gok],refA=a,floor=prof["floor_tr"][gok],
                                 realB=prof["realB_tr"][gok] if prof["realB_tr"] is not None else np.nan))
            dd["d_floor"]=dd.floor-dd.refA; dd["d_realB"]=dd.realB-dd.refA
            dd["cohort"]=coh; dd["subset"]=sub
            genediff.append(dd)
R=pd.DataFrame(rows); R.to_csv("out/rare_expression.csv",index=False)
G=pd.concat(genediff,ignore_index=True) if genediff else pd.DataFrame()
G.to_csv("out/rare_gene_diff.csv.gz",index=False,compression="gzip")
pd.set_option("display.width",280); pd.set_option("display.max_columns",40)
cols=["cohort","subset","parent","n_refA","n_realB_tr","n_floor_tr","r_realB","r_floor","nDiff_realB","nDiff_floor","mkScore_refA","mkScore_realB","mkScore_floor","auroc_refA","auroc_realB","auroc_floor"]
print(R.reindex(columns=cols).round(3).to_string(index=False))


## 6. Marker retention ratios and driver genes

In [ ]:
import numpy as np, pandas as pd
R=pd.read_csv("out/rare_expression.csv")
R=R[R.r_floor.notna()].copy()
for nm in ["realB","floor"]:
    R[f"mkRatio_{nm}"]=R[f"mkScore_{nm}"]/R["mkScore_refA"]
    R[f"mkDetRatio_{nm}"]=R[f"mkDet_{nm}"]/R["mkDet_refA"]
R["nDiff_excess"]=R.nDiff_floor-R.nDiff_realB
pd.set_option("display.width",260)
print("=== marker-programme retention within the SAME subset label ===")
print(R[["cohort","subset","n_refA","n_realB_tr","n_floor_tr","mkScore_refA","mkRatio_realB","mkRatio_floor","mkDetRatio_realB","mkDetRatio_floor","r_realB","r_floor","nDiff_realB","nDiff_floor"]].round(3).to_string(index=False))
print()
print("median marker-score ratio: realB %.2f  floor %.2f"%(R.mkRatio_realB.median(),R.mkRatio_floor.median()))
print("median marker-detection ratio: realB %.2f  floor %.2f"%(R.mkDetRatio_realB.median(),R.mkDetRatio_floor.median()))
print("median profile r: realB %.4f  floor %.4f"%(R.r_realB.median(),R.r_floor.median()))
print("median n genes |delta|>=0.5: realB %.0f  floor %.0f"%(R.nDiff_realB.median(),R.nDiff_floor.median()))
R.to_csv("out/rare_expression_ratios.csv",index=False)

G=pd.read_csv("out/rare_gene_diff.csv.gz")
print("\n=== genes most depleted / enriched in floor within each rare subset (|d|, vs real A; real B shown as control) ===")
for (coh,sub),d in G.groupby(["cohort","subset"]):
    d=d.dropna(subset=["d_floor"])
    if len(d)==0: continue
    lo=d.nsmallest(8,"d_floor"); hi=d.nlargest(8,"d_floor")
    print(f"\n-- {coh} / {sub}")
    print("   down in floor:", ", ".join(f"{r.gene}({r.d_floor:+.2f}|B{r.d_realB:+.2f})" for r in lo.itertuples()))
    print("   up   in floor:", ", ".join(f"{r.gene}({r.d_floor:+.2f}|B{r.d_realB:+.2f})" for r in hi.itertuples()))


## 7. Power-matched comparison with a real-A-internal noise floor

In [ ]:
import numpy as np, pandas as pd
from rare_lib import *
rng=np.random.default_rng(99)
rows=[]
for coh in ["sepsis","covid"]:
    X,V,SRC,genes,(DEF,PAR)=load_coh(coh)
    mA,fA=V["refA"]; detA=np.asarray((X["refA"]>0).mean(0)).ravel(); gok=detA>=0.01
    for sub,fn in DEF.items():
        idx={}
        parA=fn(mA,fA); idx["refA"]=np.where(parA&(fA==sub))[0]
        for lab in ["realB_tr","floor_tr"]:
            m,f=V[lab]; p=fn(m,f); idx[lab]=np.where(p&(f==sub))[0]
        n=min(len(v) for v in idx.values())
        if n<40: continue
        res=dict(cohort=coh,subset=sub,n_matched=n,n_refA=len(idx["refA"]),n_realB=len(idx["realB_tr"]),n_floor=len(idx["floor_tr"]))
        reps=[]
        for _ in range(10):
            pr={}
            for lab,key in [("refA","refA"),("realB_tr","realB"),("floor_tr","floor")]:
                s=rng.choice(idx[lab],n,replace=False)
                pr[key]=np.asarray(X[SRC.get(lab,"refA")][s].mean(0)).ravel()[gok] if lab!="refA" else np.asarray(X["refA"][s].mean(0)).ravel()[gok]
            # internal real A split for the true noise floor
            s2=rng.permutation(idx["refA"])
            if len(s2)>=2*n:
                a1=np.asarray(X["refA"][s2[:n]].mean(0)).ravel()[gok]; a2=np.asarray(X["refA"][s2[n:2*n]].mean(0)).ravel()[gok]
                reps.append((np.corrcoef(a1,a2)[0,1],(np.abs(a2-a1)>=0.5).sum(),
                             np.corrcoef(pr["refA"],pr["realB"])[0,1],(np.abs(pr["realB"]-pr["refA"])>=0.5).sum(),
                             np.corrcoef(pr["refA"],pr["floor"])[0,1],(np.abs(pr["floor"]-pr["refA"])>=0.5).sum()))
            else:
                reps.append((np.nan,np.nan,np.corrcoef(pr["refA"],pr["realB"])[0,1],(np.abs(pr["realB"]-pr["refA"])>=0.5).sum(),
                             np.corrcoef(pr["refA"],pr["floor"])[0,1],(np.abs(pr["floor"]-pr["refA"])>=0.5).sum()))
        a=np.array(reps,float)
        res.update(r_AA=np.nanmean(a[:,0]),nD_AA=np.nanmean(a[:,1]),r_realB=np.nanmean(a[:,2]),nD_realB=np.nanmean(a[:,3]),
                   r_floor=np.nanmean(a[:,4]),nD_floor=np.nanmean(a[:,5]))
        rows.append(res)
P=pd.DataFrame(rows); P.to_csv("out/rare_power_matched.csv",index=False)
pd.set_option("display.width",250)
print(P.round(3).to_string(index=False))
print("\nmedians: r_AA %.4f r_realB %.4f r_floor %.4f | nD_AA %.0f nD_realB %.0f nD_floor %.0f"%(
 P.r_AA.median(),P.r_realB.median(),P.r_floor.median(),P.nD_AA.median(),P.nD_realB.median(),P.nD_floor.median()))


## Outputs

| file | contents |
|---|---|
| `out/deg_summary.csv` | per cohort x six-type: DEG counts, logFC r, slope, recall/precision, top-100 overlap |
| `out/real_real_ceiling.csv` | split-half real-vs-real ceiling for the same metrics |
| `out/deg_rank2000_summary.csv` | sign agreement / slope restricted to real-expression rank > 2000, generated top-100 precision |
| `out/named_genes.csv` | per-gene real vs generated logFC for the manuscript's named genes |
| `out/rare_expression.csv`, `out/rare_expression_ratios.csv` | per rare subset: cell counts, profile r, marker score / detection, identity AUROC |
| `out/rare_gene_diff.csv.gz` | per-gene mean-expression difference vs real A inside each rare subset |
| `out/rare_power_matched.csv` | equal-n comparison with the real-A-internal noise floor |
